# Experiment Report — M2 QA Ablations

This notebook reads the output directory produced by `scripts/run_qa_ablation.py` and renders the tables and figures needed for the extended paper: zero-shot vs fine-tuning, dataset-variant ablation, model ablation, question-type breakdown, SQuAD v2 HasAns/NoAns metrics, gold-audit evaluation, and per-epoch training curves.

## 1. Configuration

Set `OUT_DIR` below to the experiment output folder (e.g. `out_experiments/run1`). Everything in this notebook is read from that folder; nothing is re-trained here.

In [ ]:
import os
import glob
import json
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# EDIT THIS: path to the experiment output directory
OUT_DIR = "out_experiments/run1"
REPORT_DIR = os.path.join(OUT_DIR, "report")
os.makedirs(REPORT_DIR, exist_ok=True)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

def load_all_metrics():
    rows = []
    for path in sorted(glob.glob(os.path.join(OUT_DIR, "**", "metrics_summary.json"), recursive=True)):
        with open(path, "r", encoding="utf-8") as f:
            rows.append(json.load(f))
    return pd.DataFrame(rows)

df = load_all_metrics()
print(f"Loaded {len(df)} experiment summaries from {OUT_DIR}")
df.head()

## 2. Main results: Zero-shot vs Fine-tuning

Overall F1 and Exact Match per dataset and mode, plus the full per-run matrix.

In [ ]:
if not df.empty:
    main_f1 = df.pivot_table(index=["dataset"], columns=["mode"], values="f1", aggfunc="mean").round(4)
    display(main_f1)
    main_f1.to_csv(os.path.join(REPORT_DIR, "main_f1_by_dataset.csv"))
    with open(os.path.join(REPORT_DIR, "table_main_f1.tex"), "w", encoding="utf-8") as f:
        f.write(main_f1.to_latex(float_format="%.4f",
                                caption="Overall F1 by dataset and mode (mean over models)",
                                label="tab:main_f1"))

    full = df[["model", "dataset", "mode", "exact", "f1",
               "HasAns_exact", "HasAns_f1", "NoAns_exact", "NoAns_f1"]].sort_values(
                   ["mode", "dataset", "model"])
    display(full.round(4))
    full.to_csv(os.path.join(REPORT_DIR, "metrics_summary_all.csv"), index=False)
    with open(os.path.join(REPORT_DIR, "table_metrics_all.tex"), "w", encoding="utf-8") as f:
        f.write(full.round(4).to_latex(index=False, caption="All experiment metrics", label="tab:all"))
else:
    print("No metrics found. Check OUT_DIR.")

In [ ]:
if not df.empty:
    # F1 heatmap: model x dataset, fine-tuning only
    ft = df[df["mode"] == "ft"]
    if not ft.empty:
        heat = ft.pivot_table(index="model", columns="dataset", values="f1", aggfunc="mean").round(4)
        fig, ax = plt.subplots(figsize=(9, 5))
        sns.heatmap(heat, annot=True, fmt=".4f", cmap="viridis", ax=ax)
        ax.set_title("Fine-tuned F1 by model and dataset")
        fig.tight_layout()
        fig.savefig(os.path.join(REPORT_DIR, "fig_heatmap_ft_f1.png"), bbox_inches="tight")
        fig.savefig(os.path.join(REPORT_DIR, "fig_heatmap_ft_f1.pdf"), bbox_inches="tight")
        plt.show()

    # ZSL vs FT delta
    if {"zsl", "ft"}.issubset(set(df["mode"])):
        piv = df.pivot_table(index=["model", "dataset"], columns="mode", values=["f1", "exact"]).reset_index()
        piv.columns = ["_".join(c) if isinstance(c, tuple) else c for c in piv.columns]
        piv["f1_delta"] = piv["f1_ft"] - piv["f1_zsl"]
        piv["exact_delta"] = piv["exact_ft"] - piv["exact_zsl"]
        display(piv.round(4))
        piv.to_csv(os.path.join(REPORT_DIR, "zsl_vs_ft_delta.csv"), index=False)
        fig, ax = plt.subplots(figsize=(10, 4))
        sns.barplot(data=piv, x="dataset", y="f1_delta", hue="model", ax=ax)
        ax.set_title("F1 improvement of fine-tuning over zero-shot")
        ax.axhline(0, color="gray", linestyle="--")
        fig.tight_layout()
        fig.savefig(os.path.join(REPORT_DIR, "fig_f1_delta.png"), bbox_inches="tight")
        fig.savefig(os.path.join(REPORT_DIR, "fig_f1_delta.pdf"), bbox_inches="tight")
        plt.show()

## 3. Question-type breakdown

EM/F1 per question kind (time, date, value, place, objects) aggregated over all runs.

In [ ]:
kind_rows = []
for path in sorted(glob.glob(os.path.join(OUT_DIR, "**", "metrics_by_question_type.csv"), recursive=True)):
    d = pd.read_csv(path)
    rel = os.path.relpath(path, OUT_DIR).split(os.sep)
    if len(rel) >= 3:
        d["model"] = rel[0]
        d["dataset"] = rel[1]
        d["mode"] = rel[2]
    kind_rows.append(d)

if kind_rows:
    kdf = pd.concat(kind_rows, ignore_index=True)
    kp = kdf.pivot_table(index=["kind"], columns=["mode"], values="f1", aggfunc="mean").round(4)
    display(kp)
    kp.to_csv(os.path.join(REPORT_DIR, "f1_by_question_type.csv"))
    with open(os.path.join(REPORT_DIR, "table_f1_by_question_type.tex"), "w", encoding="utf-8") as f:
        f.write(kp.to_latex(float_format="%.4f",
                           caption="F1 by question type and mode (mean over models/datasets)",
                           label="tab:qtype"))
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(data=kdf, x="kind", y="f1", hue="mode", ax=ax)
    ax.set_title("F1 by question type (all models/datasets)")
    fig.tight_layout()
    fig.savefig(os.path.join(REPORT_DIR, "fig_f1_by_question_type.png"), bbox_inches="tight")
    fig.savefig(os.path.join(REPORT_DIR, "fig_f1_by_question_type.pdf"), bbox_inches="tight")
    plt.show()
else:
    print("No question-type metrics found.")

## 4. HasAns / NoAns metrics (SQuAD v2)

Overall, answerable, and unanswerable EM/F1 per mode. This is the part that extends the original answerable-only paper.

In [ ]:
if not df.empty:
    noans = df.groupby("mode")[["exact", "f1", "HasAns_exact", "HasAns_f1",
                              "NoAns_exact", "NoAns_f1"]].mean().round(4)
    display(noans)
    noans.to_csv(os.path.join(REPORT_DIR, "hasans_noans_summary.csv"))
    with open(os.path.join(REPORT_DIR, "table_hasans_noans.tex"), "w", encoding="utf-8") as f:
        f.write(noans.to_latex(float_format="%.4f",
                              caption="SQuAD v2 overall / HasAns / NoAns metrics by mode",
                              label="tab:noans"))
    m = df.melt(id_vars=["mode"], value_vars=["HasAns_f1", "NoAns_f1", "f1"],
                var_name="metric", value_name="score")
    fig, ax = plt.subplots(figsize=(9, 5))
    sns.barplot(data=m, x="metric", y="score", hue="mode", ax=ax)
    ax.set_title("HasAns / NoAns / overall F1 by mode")
    fig.tight_layout()
    fig.savefig(os.path.join(REPORT_DIR, "fig_hasans_noans.png"), bbox_inches="tight")
    fig.savefig(os.path.join(REPORT_DIR, "fig_hasans_noans.pdf"), bbox_inches="tight")
    plt.show()

## 5. Gold-audit evaluation

Metrics on the 200 GLM-labeled rows (`audit_stratified_sample_labeled_v1.csv`), using `gold_answer`/`gold_is_impossible` as references. The sample is stratified, so treat these as a pilot gold test, not population estimates.

In [ ]:
gold_rows = []
for path in sorted(glob.glob(os.path.join(OUT_DIR, "**", "gold_audit_metrics.json"), recursive=True)):
    with open(path, "r", encoding="utf-8") as f:
        gold_rows.append(json.load(f))

if gold_rows:
    gdf = pd.DataFrame(gold_rows)
    cols = [c for c in ["model", "dataset", "mode", "exact", "f1",
                        "HasAns_exact", "HasAns_f1", "NoAns_exact", "NoAns_f1"] if c in gdf.columns]
    display(gdf[cols].round(4))
    gdf.to_csv(os.path.join(REPORT_DIR, "gold_audit_metrics.csv"), index=False)
    with open(os.path.join(REPORT_DIR, "table_gold_audit.tex"), "w", encoding="utf-8") as f:
        f.write(gdf[cols].round(4).to_latex(index=False, caption="Gold-audit metrics by run", label="tab:gold"))
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.barplot(data=gdf, x="dataset", y="f1", hue="mode", ax=ax)
    ax.set_title("Gold-audit F1 by dataset and mode (mean over models)")
    fig.tight_layout()
    fig.savefig(os.path.join(REPORT_DIR, "fig_gold_audit_f1.png"), bbox_inches="tight")
    fig.savefig(os.path.join(REPORT_DIR, "fig_gold_audit_f1.pdf"), bbox_inches="tight")
    plt.show()
else:
    print("No gold-audit metrics found.")

## 6. Training curves

Per-epoch training/validation loss and validation EM/F1 for every fine-tuned run, to decide how many epochs are appropriate.

In [ ]:
hist_paths = sorted(glob.glob(os.path.join(OUT_DIR, "**", "training_history.csv"), recursive=True))
print(f"Found {len(hist_paths)} training histories")

if hist_paths:
    n = len(hist_paths)
    cols = min(3, n)
    rows = math.ceil(n / cols) if cols else 0

    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows), squeeze=False)
    for ax, path in zip(axes.ravel(), hist_paths):
        h = pd.read_csv(path)
        rel = os.path.relpath(path, OUT_DIR).split(os.sep)
        ax.plot(h["epoch"], h["train_loss"], marker="o", label="train loss")
        ax.plot(h["epoch"], h["eval_loss"], marker="o", label="eval loss")
        ax.set_title(" / ".join(rel[:3]), fontsize=10)
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss")
        ax.legend(fontsize=8)
    for ax in axes.ravel()[len(hist_paths):]:
        ax.axis("off")
    fig.tight_layout()
    fig.savefig(os.path.join(REPORT_DIR, "fig_training_curves_loss.png"), bbox_inches="tight", dpi=150)
    fig.savefig(os.path.join(REPORT_DIR, "fig_training_curves_loss.pdf"), bbox_inches="tight")
    plt.show()

    fig2, axes2 = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows), squeeze=False)
    for ax, path in zip(axes2.ravel(), hist_paths):
        h = pd.read_csv(path)
        rel = os.path.relpath(path, OUT_DIR).split(os.sep)
        ax.plot(h["epoch"], h["eval_exact"], marker="o", label="eval EM")
        ax.plot(h["epoch"], h["eval_f1"], marker="o", label="eval F1")
        ax.set_title(" / ".join(rel[:3]), fontsize=10)
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Score")
        ax.legend(fontsize=8)
    for ax in axes2.ravel()[len(hist_paths):]:
        ax.axis("off")
    fig2.tight_layout()
    fig2.savefig(os.path.join(REPORT_DIR, "fig_training_curves_scores.png"), bbox_inches="tight", dpi=150)
    fig2.savefig(os.path.join(REPORT_DIR, "fig_training_curves_scores.pdf"), bbox_inches="tight")
    plt.show()

## 7. Notes

- All CSVs, LaTeX tables, and figures are exported to `OUT_DIR/report/`.
- Gold-audit numbers come from the 200 GLM-labeled rows and are stratified; report them with the caveat that they are a pilot gold set.
- The common test split is context-level (10% dev / 10% test, seed 42), so all dataset variants are evaluated on the same references.